# Residual Flow Analysis

Track how information flows through the residual stream: direction,
norms, component contributions, signal-noise decomposition, and
cross-position flow.

## Why This Matters

Residual stream flow characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_flow_analysis import (
    residual_direction_flow, residual_norm_flow,
    residual_component_flow, residual_signal_noise,
    residual_cross_position_flow,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Direction Flow

How much does the residual stream direction change layer-by-layer?

In [ ]:
result = residual_direction_flow(model, tokens)
print(f"Embed→Final cosine: {result['embed_to_final_cosine']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int((1 - p['direction_change']) * 20)
    print(f"  {str(p['from_layer']):5s} → L{p['to_layer']}: "
          f"cos={p['direction_cosine']:.4f}, change={p['direction_change']:.4f} {bar}")

## Norm Flow

Track norm growth or decay through the residual stream.

In [ ]:
result = residual_norm_flow(model, tokens)
print(f"Embed norm: {result['embed_norm']:.4f} → Final: {result['final_norm']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int(min(p['growth_factor'], 3) * 5)
    print(f"  Layer {p['layer']}: norm={p['norm']:.4f}, "
          f"growth={p['growth_factor']:.2f}x, cum={p['cumulative_growth']:.2f}x {bar}")

## Component Flow

Attention vs MLP contributions at each layer.

In [ ]:
result = residual_component_flow(model, tokens)
for p in result['per_layer']:
    attn_bar = '▓' * int(p['attn_fraction'] * 20)
    mlp_bar = '░' * int(p['mlp_fraction'] * 20)
    print(f"  Layer {p['layer']}: attn={p['attn_fraction']:.1%} mlp={p['mlp_fraction']:.1%} "
          f"cos={p['attn_mlp_cosine']:.3f} {attn_bar}{mlp_bar}")

## Signal-Noise Decomposition

Decompose residual into signal (aligned with final output) and noise.

In [ ]:
result = residual_signal_noise(model, tokens)
print(f"Embed signal: {result['embed_signal']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: signal={p['signal']:.4f}, "
          f"noise={p['noise']:.4f}, SNR={p['snr']:.3f}")

## Cross-Position Flow

How similar are residual streams across different positions?

In [ ]:
result = residual_cross_position_flow(model, tokens)
print(f"Diverse: {result['is_diverse']} (mean sim: {result['mean_pairwise_similarity']:.4f})\n")
for p in result['per_position']:
    bar = '█' * int(max(0, p['mean_similarity_to_others']) * 20)
    print(f"  Pos {p['position']}: norm={p['norm']:.4f}, "
          f"mean_sim={p['mean_similarity_to_others']:.4f} {bar}")